### Importing packages

In [1]:
import pandas as pd              
import numpy as np             

### Importing dataset

In [2]:
location = "zambia"
dataset = "test"
parent_path = '/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Zambia/64x64/full'

# Load data
data_X = pd.read_csv(f'{parent_path}/{dataset}/{"train" if dataset == "train" else "test"}-data-{location}-input-t0.csv')
target_t0 = pd.read_csv(f'{parent_path}/{dataset}/{"train" if dataset == "train" else "test"}-data-{location}-output-map-t0.csv', header=None)
target = pd.read_csv(f'{parent_path}/{dataset}/{"train" if dataset == "train" else "test"}-data-{location}-output-map-t1.csv', header=None)

### Choosing lead time

In [3]:
lead_time = 6

### Choosing data split

In [4]:
original_data = data_X.copy()

### Finding exact rows after given time (hh, mm)

In [5]:
data_X['datetime'] = pd.to_datetime(data_X[['year', 'month', 'day', 'hour', 'minute']])

In [6]:
def find_exact_row_after_given_hours(row, hours, minutes, df):
    target_time = row['datetime'] + pd.Timedelta(hours=hours, minutes=minutes)
    corresponding_row = df[df['datetime'] == target_time]
    if not corresponding_row.empty:
        return corresponding_row.index[0]  # Return the index of the corresponding row
    else:
        return None

In [7]:
data_X['row_index_X0_60'] = data_X.apply(find_exact_row_after_given_hours, args=(1, 0, data_X), axis=1)
data_X['row_index_X0_120'] = data_X.apply(find_exact_row_after_given_hours, args=(2, 0, data_X), axis=1)

One hour lead time

This is relative to 120 as it's the nowcast origin

+1 as $t_0$ is at 120 minutes

### Find the target index for the chosen lead time for each row

We need $t_0$, $t_1$ and $t_{lt}$

This it to get the corresponding target value after one hour

In [8]:
data_X['row_index_Cb'] = data_X.apply(find_exact_row_after_given_hours, args=(lead_time + 1 , 0, data_X), axis=1)      # since the target is at t0+1 h

In [9]:
columns_to_keep = original_data.keys().to_list()

In [10]:
# Function to combine current row with rows based on indices, retaining original column order
def combine_current_and_rows(row, data):
    # Extract the row indices from the current row
    indices = [row['row_index_X0_60'], row['row_index_X0_120']]
    
    # Fetch the current row (filter only relevant columns)
    current_row = row[columns_to_keep].copy()
    
    # List to store the rows
    rows_to_combine = [current_row]
    
    # Fetch rows corresponding to the indices, rename columns with suffix to avoid duplicates
    for i, idx in enumerate(indices):
        if pd.notna(idx):
            # Fetch the row, keep only relevant columns, and rename them with suffix
            fetched_row = data.loc[idx, columns_to_keep].rename(lambda col: f"{col}_{(i+1)*60}")
            rows_to_combine.append(fetched_row)
        else:
            # If the index is NaN, create an empty Series with the same columns as the current row
            empty_row = pd.Series(index=[f"{col}_{(i+1)*60}" for col in columns_to_keep])
            rows_to_combine.append(empty_row)
    
    # Concatenate the current row with the fetched rows side by side (maintain column order)
    combined_row = pd.concat(rows_to_combine, axis=0)
    
    return combined_row

    
# Apply the function row by row to get combined data
combined_data = data_X.apply(combine_current_and_rows, args=(data_X,), axis=1)

# Convert the combined series into a DataFrame while keeping the original index
combined_df = pd.DataFrame(combined_data, index=data_X.index)

### Taking target values

In [11]:
combined_df = pd.DataFrame(combined_data, index=data_X.index)

In [12]:
combined_df = combined_df.dropna()

In [13]:
combined_df.keys()

Index(['year', 'month', 'day', 'hour', 'minute', 'lat1', 'lat2', 'lon1',
       'lon2', 'wp1', 'wp2', 'size1', 'size2', 'd1', 'd2', 'mask1', 'mask2',
       'year_60', 'month_60', 'day_60', 'hour_60', 'minute_60', 'lat1_60',
       'lat2_60', 'lon1_60', 'lon2_60', 'wp1_60', 'wp2_60', 'size1_60',
       'size2_60', 'd1_60', 'd2_60', 'mask1_60', 'mask2_60', 'year_120',
       'month_120', 'day_120', 'hour_120', 'minute_120', 'lat1_120',
       'lat2_120', 'lon1_120', 'lon2_120', 'wp1_120', 'wp2_120', 'size1_120',
       'size2_120', 'd1_120', 'd2_120', 'mask1_120', 'mask2_120'],
      dtype='object')

In [14]:
combined_df[['year', 'month', 'day', 'hour', 'minute', 'year_60', 'month_60', 'day_60', 'hour_60', 'minute_60', 'year_120', 'month_120', 'day_120', 'hour_120', 'minute_120']]

,year,month,day,hour,minute,year_60,month_60,day_60,hour_60,minute_60,year_120,month_120,day_120,hour_120,minute_120
0,2020,1,1,0,0,2020.0,1.0,1.0,1.0,0.0,2020.0,1.0,1.0,2.0,0.0
1,2020,1,1,0,15,2020.0,1.0,1.0,1.0,15.0,2020.0,1.0,1.0,2.0,15.0
2,2020,1,1,0,30,2020.0,1.0,1.0,1.0,30.0,2020.0,1.0,1.0,2.0,30.0
3,2020,1,1,0,45,2020.0,1.0,1.0,1.0,45.0,2020.0,1.0,1.0,2.0,45.0
4,2020,1,1,1,0,2020.0,1.0,1.0,2.0,0.0,2020.0,1.0,1.0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33002,2024,1,25,21,30,2024.0,1.0,25.0,22.0,30.0,2024.0,1.0,25.0,23.0,30.0
33003,2024,1,25,21,45,2024.0,1.0,25.0,22.0,45.0,2024.0,1.0,25.0,23.0,45.0
33005,2024,1,25,22,15,2024.0,1.0,25.0,23.0,15.0,2024.0,1.0,26.0,0.0,15.0
33006,2024,1,25,22,30,2024.0,1.0,25.0,23.0,30.0,2024.0,1.0,26.0,0.0,30.0


In [15]:
target_index = data_X['row_index_Cb'][combined_df.index].dropna()
target = target.loc[target_index]
target_t0 = target_t0.loc[target_index]
combined_df = combined_df.loc[target_index.index]

In [16]:
target = target.to_numpy()
target_t0 = target_t0.to_numpy()

In [17]:
target.shape

(31619, 4096)

In [18]:
resolution_y, resolution_x = 64, 64
target = target.reshape(len(target), resolution_y, resolution_x)
target_t0 = target_t0.reshape(len(target_t0), resolution_y, resolution_x)

In [19]:
combined_df

,year,month,day,hour,minute,lat1,lat2,lon1,lon2,wp1,...,lon1_120,lon2_120,wp1_120,wp2_120,size1_120,size2_120,d1_120,d2_120,mask1_120,mask2_120
0,2020,1,1,0,0,-13.389998,-14.328512,32.175030,32.314232,128.861111,...,31.391163,24.836910,96.928571,265.487912,126.0,8190.0,383.538052,478.073484,1.0,1.0
1,2020,1,1,0,15,-13.357220,-14.644295,32.025070,32.049854,103.500000,...,24.803896,33.087578,263.446242,116.785714,9459.0,378.0,480.547348,552.647276,1.0,1.0
2,2020,1,1,0,30,-14.322840,-16.317184,32.095791,25.049038,118.723404,...,24.742725,33.056164,259.189775,102.608696,10386.0,207.0,487.418538,549.498855,1.0,1.0
3,2020,1,1,0,45,-14.346675,-14.030564,31.883835,32.112480,107.826087,...,26.490522,24.582918,314.140845,242.330211,639.0,11529.0,409.614000,501.847188,1.0,1.0
4,2020,1,1,1,0,-14.343847,-14.027892,31.775215,32.003929,99.432432,...,26.490522,24.587486,341.111940,211.300725,1206.0,12420.0,409.614000,503.703206,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32998,2024,1,25,20,30,-3.568615,-5.614813,29.251968,19.359585,155.788043,...,25.023991,23.185144,139.307692,153.900000,117.0,180.0,429.716792,1306.132767,1.0,1.0
32999,2024,1,25,20,45,-3.568615,-5.532207,29.251968,19.415659,154.892086,...,25.053329,23.213373,134.000000,168.050000,315.0,540.0,429.379406,1318.957659,1.0,1.0
33000,2024,1,25,21,0,-3.540682,-5.477293,29.250862,19.473000,145.686957,...,25.018114,23.212280,108.027778,137.304348,324.0,828.0,434.397181,1324.650313,1.0,1.0
33001,2024,1,25,21,15,-3.568802,-5.504746,29.285330,19.444309,125.012048,...,23.242550,23.468369,131.500000,0.000000,954.0,0.0,1326.146535,1901.699454,1.0,0.0


In [20]:
output_dir = '/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Data/Zambia/64x64/full/timesteps'

In [21]:
combined_df.to_csv(f'{output_dir}/{dataset}-{location}-input-t0-for-lt{lead_time}.csv', index=False)

In [22]:
np.save(f'{output_dir}/{dataset}-{location}-output-at-lt{lead_time}.npy', target)
np.save(f'{output_dir}/{dataset}-{location}-output-t0-for-lt{lead_time}.npy', target_t0)